<a href="https://colab.research.google.com/github/nmhom/MAT-422/blob/main/MAT422_HW_3.7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW 3.7

This homework demonstrates the use of differentiation for training artificial neural networks to minimize cost functions.

In [7]:
import numpy as np

## 3.7.1. Mathematical formulation

A neural network takes inputs, multiplies them by weights, adds a bias, and applies an activation function to calculate its output. Each layer passes its output to the next layer, During training, the weights and biases are adjusted to make the predictions more accurate.

For a simple neural network with two inputs: $$z = w_1x_1 + w_2x_2 + b$$
$$\hat{y} = \sigma(z)$$

Where:
- $x_1$, $x_2$: input values
- $w_1$, $w_2$: weights that determine how strongly each input influences the result
- b: bias, a constant added to the weighted sum
- z: weighted sum before activation
- $\sigma$: activation function
- $\hat{y}$: predicted output

For a general neural network, the equations become: $$z^{(l)} = W^{(l)}a^{(l-1)} + b^{(l)}$$ $$a^{(l)} = \sigma(z^{(l)})$$

Here $l$ represents the current layer, $a^{l-1}$ contains the outputs from the previous layer, and $w^{(l)}$ and $b^{(l)}$ are the current layer's weights and biases.

In [8]:
# Two input features
x = np.array([2,3])

# Weights and bias
w = np.array([0.5, 0.2])
b = 0.1

# Calculate the weighted sum
z = np.dot(w,x) + b

print("weighted sum:", z)

weighted sum: 1.7000000000000002


This demonstrates how inputs, weights, abd bias combine before the activation function is applied.

## 3.7.2. Activation functions

Activation functions introduce different behaviors into a neural network. The step function makes a binary decision, ReLU allows positive values to pass through, sigmoid produces a probability-like output, and softmax produces a probability distribution over multiple classes.

Activation functions transform the weighted sum z into the ouput of a neuron.
1. Step function

$$
\sigma(x)=
\begin{cases}
0, & x<0\\
1, & x\geq 0
\end{cases}
\
$$

Produces either 0 or 1.

2. ReLU (Rectified Linear Unit)

$$\sigma(x)=\max(0,x)$$

Returns 0 for negative inputs and leaves positive inputs unchanged.

3. Sigmoid

$$
\sigma(x)=\frac{1}{1+e^{-x}}
\$$

Maps any real-valued input to a number between 0 and 1. This makes it useful for predicting probabilities in binary classification.

4. Softmax

$$
\
\sigma(z_k)=\frac{e^{z_k}}{\sum_{j=1}^{K}e^{z_j}}
\
$$

Converts a vector of scores into probabilities that sum to 1, commonly for classification into multiple categories.

In [9]:
z = np.array([-2.0, 0.0, 2.0])

# Step function
step = (z>=0).astype(int)

# ReLU
relu = np.maximum(0, z)

# Sigmoid
sigmoid = 1 / (1 + np.exp(-z))

# Softmax
softmax = np.exp(z) / np.sum(np.exp(z))

print("Step:", step)
print("ReLU:", relu)
print("Sigmoid:", sigmoid)
print("Softmax:", softmax)
print("Softmax sum:", np.sum(softmax))

Step: [0 1 1]
ReLU: [0. 0. 2.]
Sigmoid: [0.11920292 0.5        0.88079708]
Softmax: [0.01587624 0.11731043 0.86681333]
Softmax sum: 1.0


## 3.7.3. Cost function

A cost function measures how well the network performs. Smaller cost generally means predictions are closer to the target values. Training aims to find weights and biases that minimizes this cost. Quadratic cost is useful for demonstrating numerical error, while binary cross-entropy is commonly used for predicting one of two classes.

The cost function measures how different the predicted outputs are from the actual outputs.

1. Quadratic cost (least squares)
$$
\
J=\frac{1}{2}\sum_{n=1}^{N}\sum_{k=1}^{K}
\left(\hat{y}^{(n)}_k-y^{(n)}_k\right)^2
\
$$

Where:
- N: number of training examples
- K: number of output nodes
- $\hat{y}$: predicted output
- y: actual output

The factor 1/2 makes the derivative simpler

2. Binary cross-entropy
$$
\
J=-\sum_{n=1}^{N}
\left[
y^{(n)}\ln\hat{y}^{(n)}
+(1-y^{(n)})\ln(1-\hat{y}^{(n)})
\right]
\
$$

This is commonly used for binary classification when the prediction $\hat{y}$ is a probability between 0 and 1.



In [10]:
# Actual values and predicted probabilities
y = np.array([1,0,1])
y_hat = np.array([0.8, 0.3, 0.6])

# Quadratic cost
quadratic_cost = 0.5 * np.sum((y_hat - y) ** 2)

# Binary cross-entropy
epsilon = 1e-10
p = np.clip(y_hat, epsilon, 1 - epsilon)
binary_cross_entropy = -np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))

print("Quadratic cost:", quadratic_cost)
print("Binary cross-entropy:", binary_cross_entropy)

Quadratic cost: 0.14500000000000002
Binary cross-entropy: 1.0906441190189329


## 3.7.4. Backpropagation

Backpropagation determines how much weight and bias contributes to the prediction error. Using the chain rule, it calculates the cost gradients, starting at the output and propagating backward through the hidden layers. These gradients tell the training algorithm how to change the parameters to reduce the cost.

First, define the error quantity:
$$
\
\delta^{(l)}_{j'}=
\frac{\partial J}{\partial z^{(l)}_{j'}}
\
$$

For a neuron in the output layer, the quadratic-cost error is:

$$
\delta^{(L)}_{j'}=
\left(\hat y_{j'}-y_{j'}\right)
\sigma'\left(z^{(L)}_{j'}\right)
\
$$

For a hidden layer, the error is propagated backward:

$$
\
\delta^{(l-1)}_j=
\sigma'\left(z^{(l-1)}_j\right)
\sum_{j'}\delta^{(l)}_{j'}w^{(l)}_{j,j'}
\
$$

The derivative of the cost with respect to a weight is:
$$
\
\frac{\partial J}{\partial w^{(l)}_{j,j'}}
=\delta^{(l)}_{j'}a^{(l-1)}_j
\
$$

The derivative with respect to a bias is:

$$
\
\frac{\partial J}{\partial b^{(l)}_{j'}}
=\delta^{(l)}_{j'}
\
$$

Useful activation derivatives include:

ReLU
$$
\
\sigma'(z)=
\begin{cases}
0,&z<0\\
1,&z>0
\end{cases}
\
$$

Sigmoid

$$\
\sigma'(z)=\sigma(z)(1-\sigma(z))
\$$

At z = 0, ReLU's derivative is conventionally chosen as 0 in many implementations.

In [11]:
x = 2.0
y = 1.0
w = 0.5
b = 0.1

# Forward pass
z = w * x + b
y_hat = 1 / (1 + np.exp(-z))

# Quadratic cost
J = 0.5 * (y_hat - y) ** 2

# Backpropagation
sigmoid_derivative = y_hat * (1- y_hat)
delta = (y_hat - y) * sigmoid_derivative

dJ_dw=delta * x # How the cost changes when the weight changes
dJ_db= delta # how the cost changes when the bias changes

print("Prediction:", y_hat)
print("Cost:", J)
print("Gradient with respect to weight:", dJ_dw)
print("Gradient with respect to bias:", dJ_db)

Prediction: 0.7502601055951177
Cost: 0.031185007428680883
Gradient with respect to weight: -0.0935874678657066
Gradient with respect to bias: -0.0467937339328533


## 3.7.5. Backpropagation algorithm

The backpropagation algorithm trains a neural network by repeatedly calculating predictions, measuring the cost, computing gradients, and updating weights and biases with gradient descent. Over repeated iterations, the network can learn parameters that produce more accurate predictions.

The algorithm combines forward propagation, backpropagation, and gradient descent.

Step 1: Forward propagation

Calculate the weighted sum and activation for each layer.
$$z^{(l)}=W^{(l)}a^{(l-1)}+b^{(l)}$$
$$a^{(l)}=\sigma(z^{(l)})$$

Calculate the network's output and its cost.

Step 2: Calculate the gradients

The gradient with respect to weight is:
$$\frac{\partial J}{\partial w^{(l)}_{j,j'}}
=\delta^{(l)}_{j'}a^{(l-1)}_j$$

The gradient with respect to a bias is:

$$\frac{\partial J}{\partial b^{(l)}_{j'}}
=\delta^{(l)}_{j'}$$

Step 3: Update the weights and biases
$$
w^{(l)}{\text{new}}
= w^{(l)}{\text{old}} - \beta\frac{\partial J}{\partial w^{(l)}}
$$

$$b^{(l)}{\text{new}} = b^{(l)}{\text{old}} - \beta\frac{\partial J}{\partial b^{(l)}}$$

Here, $\beta$ is the learning rate, which controls the size of each update.

Step 4: Repeat

Repeat the forward pass, gradient calculations, and parameter updates until the cost is sufficiently small or another stopping criterion is reached.

In [12]:
import numpy as np

x = 2.0
y = 1.0

# Initial parameters
w = 0.5
b = 0.1

learning_rate = 0.5

for epoch in range(100):
    # Forward pass
    z = w * x + b
    y_hat = 1 / (1 + np.exp(-z))

    # Cost
    J = 0.5 * (y_hat - y) ** 2

    # Backpropagation
    delta = (y_hat - y) * y_hat * (1 - y_hat)
    dJ_dw = delta * x
    dJ_db = delta

    # Gradient descent
    w -= learning_rate * dJ_dw
    b -= learning_rate * dJ_db

print("Final prediction:", y_hat)
print("Final cost:", J)
print("Final weight:", w)
print("Final bias:", b)

Final prediction: 0.9522789568221364
Final cost: 0.0011386489809917626
Final weight: 1.259562845471189
Final bias: 0.4797814227355945
